# Session-based Recommendation with XLNet — CPU

Replicates the Transformers4Rec XLNet-MLM architecture using only `transformers==4.30.1` and `torch` (CPU only).  
No dependency on transformers4rec or merlin.

In [ ]:
import os
import gc
import csv
import glob
import math
import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

from transformers import XLNetConfig, XLNetModel
from tqdm.auto import tqdm

device = torch.device("cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Feature configuration
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = {
    "sess_pid_seq":  {"cardinality": 164386, "embedding_dim": 64},
    "sess_csid_seq": {"cardinality": 622,    "embedding_dim": 32},
    "sess_ccid_seq": {"cardinality": 129,    "embedding_dim": 16},
    "sess_bid_seq":  {"cardinality": 3426,   "embedding_dim": 32},
}

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

ITEM_ID_FEATURE = "sess_pid_seq"

# ---------------------------------------------------------------------------
# Model hyperparameters
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 20
D_MODEL = 448
N_HEAD = 8
N_LAYER = 2
CONTINUOUS_PROJ_DIM = 64
MLM_PROBABILITY = 0.1
LABEL_SMOOTHING = 0.5

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
LEARNING_RATE = 0.00020171456712823088
WEIGHT_DECAY = 2.747484129693843e-05
NUM_EPOCHS = 10
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512

# ---------------------------------------------------------------------------
# Data paths
# ---------------------------------------------------------------------------
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "preprocessed_data"))
OUTPUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "data_trimmed"))

START_WINDOW = int(os.environ.get("start_window_index", "1"))
FINAL_WINDOW = int(os.environ.get("final_window_index", "30"))

print(f"Source data:  {SRC_DIR}")
print(f"Trimmed data: {OUTPUT_DIR}")
print(f"Window range: {START_WINDOW} \u2013 {FINAL_WINDOW}")

## Dataset

In [ ]:
class SessionDataset(Dataset):
    """Loads parquet files and converts variable-length sessions into
    fixed-length tensors (padded / truncated to MAX_SEQ_LEN)."""

    def __init__(self, parquet_paths, max_seq_len=MAX_SEQ_LEN):
        dfs = [pd.read_parquet(p) for p in parquet_paths]
        self.df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq_len = min(int(row["sess_seq_len"]), self.max_seq_len)
        result = {}

        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[: self.max_seq_len]
            padded = np.zeros(self.max_seq_len, dtype=np.int64)
            padded[: len(arr)] = arr
            result[feat_name] = torch.tensor(padded, dtype=torch.long)

        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[: self.max_seq_len]
            padded = np.zeros(self.max_seq_len, dtype=np.float32)
            padded[: len(arr)] = arr
            cont_arrays.append(padded)
        result["continuous"] = torch.tensor(
            np.stack(cont_arrays, axis=-1), dtype=torch.float32
        )

        mask = np.zeros(self.max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        result["attention_mask"] = torch.tensor(mask, dtype=torch.float32)

        result["labels"] = result[ITEM_ID_FEATURE].clone()

        return result


_sample_paths = glob.glob(os.path.join(OUTPUT_DIR, "0001/train.parquet"))
_ds = SessionDataset(_sample_paths)
_item = _ds[0]
print(f"Dataset size: {len(_ds)}")
for k, v in _item.items():
    print(f"  {k}: {v.shape} {v.dtype}")
del _ds, _item

## Model Architecture

In [ ]:
class SessionXLNetModel(nn.Module):
    """XLNet-based session recommendation model with MLM masking and
    weight-tied next-item prediction head."""

    def __init__(self):
        super().__init__()

        self.embeddings = nn.ModuleDict()
        for feat, cfg in CATEGORICAL_FEATURES.items():
            self.embeddings[feat] = nn.Embedding(
                cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
            )

        self.continuous_proj = nn.Sequential(
            nn.Linear(len(CONTINUOUS_FEATURES), CONTINUOUS_PROJ_DIM),
            nn.ReLU(),
        )

        cat_dim = sum(c["embedding_dim"] for c in CATEGORICAL_FEATURES.values())
        total_input_dim = cat_dim + CONTINUOUS_PROJ_DIM
        self.input_mlp = nn.Sequential(
            nn.Linear(total_input_dim, D_MODEL),
            nn.ReLU(),
        )

        self.mask_token = nn.Parameter(torch.randn(D_MODEL) * 0.02)

        xlnet_config = XLNetConfig(
            vocab_size=2,
            d_model=D_MODEL,
            n_layer=N_LAYER,
            n_head=N_HEAD,
            d_inner=4 * D_MODEL,
            ff_activation="gelu",
            attn_type="bi",
            mem_len=0,
            dropout=0.0,
        )
        self.xlnet = XLNetModel(xlnet_config)

        item_emb_dim = CATEGORICAL_FEATURES[ITEM_ID_FEATURE]["embedding_dim"]
        self.output_proj = nn.Linear(D_MODEL, item_emb_dim, bias=False)
        self.output_bias = nn.Parameter(
            torch.zeros(CATEGORICAL_FEATURES[ITEM_ID_FEATURE]["cardinality"])
        )

        self.loss_fn = nn.CrossEntropyLoss(
            ignore_index=0, label_smoothing=LABEL_SMOOTHING
        )

    def _item_embedding_weight(self):
        return self.embeddings[ITEM_ID_FEATURE].weight

    def forward(self, batch, training=True):
        cat_embeds = [self.embeddings[f](batch[f]) for f in CATEGORICAL_FEATURES]
        cont_proj = self.continuous_proj(batch["continuous"])

        x = torch.cat(cat_embeds + [cont_proj], dim=-1)
        x = self.input_mlp(x)

        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        if training:
            rand = torch.rand_like(attention_mask)
            mlm_mask = (rand < MLM_PROBABILITY) & (attention_mask > 0)
        else:
            seq_lens = attention_mask.sum(dim=1).long()
            mlm_mask = torch.zeros_like(attention_mask, dtype=torch.bool)
            batch_idx = torch.arange(mlm_mask.size(0), device=mlm_mask.device)
            last_pos = (seq_lens - 1).clamp(min=0)
            mlm_mask[batch_idx, last_pos] = True

        mask_expanded = mlm_mask.unsqueeze(-1)
        x = torch.where(mask_expanded, self.mask_token.expand_as(x), x)

        xlnet_out = self.xlnet(inputs_embeds=x, attention_mask=attention_mask)
        hidden = xlnet_out.last_hidden_state

        projected = self.output_proj(hidden)
        logits = F.linear(
            projected, self._item_embedding_weight(), self.output_bias
        )

        masked_labels = labels.clone()
        masked_labels[~mlm_mask] = 0
        loss = self.loss_fn(
            logits.view(-1, logits.size(-1)), masked_labels.view(-1)
        )

        return loss, logits, mlm_mask


model = SessionXLNetModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## Evaluation Metrics

In [ ]:
def compute_ranking_metrics(logits, labels, mlm_mask, top_ks=(20, 40)):
    masked_logits = logits[mlm_mask]
    masked_labels = labels[mlm_mask]

    if masked_logits.numel() == 0:
        return {}

    results = {}
    for k in top_ks:
        _, topk_idx = torch.topk(masked_logits, k, dim=-1)
        match = topk_idx == masked_labels.unsqueeze(-1)

        hits = match.any(dim=-1).float()
        results[f"recall_at_{k}"] = (hits.sum().item(), len(hits))

        positions = match.float() * torch.arange(
            1, k + 1, device=match.device, dtype=torch.float32
        )
        rank = positions.sum(dim=-1)
        dcg = torch.where(rank > 0, 1.0 / torch.log2(rank + 1),
                          torch.zeros_like(rank))
        results[f"ndcg_at_{k}"] = (dcg.sum().item(), len(dcg))

    return results

## Training & Evaluation Helpers

In [ ]:
# ---------------------------------------------------------------------------
# CSV logger
# ---------------------------------------------------------------------------
LOG_FIELDS = [
    "timestamp", "phase", "day", "epoch", "batch",
    "loss", "ndcg_at_20", "ndcg_at_40", "recall_at_20", "recall_at_40",
]


def init_log(path):
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow(LOG_FIELDS)
    return path


def log_row(path, **kwargs):
    with open(path, "a", newline="") as f:
        csv.writer(f).writerow([kwargs.get(c, "") for c in LOG_FIELDS])


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train_one_day(model, train_loader, optimizer, scheduler, log_path, day):
    model.train()
    epoch_bar = tqdm(range(NUM_EPOCHS), desc="Epochs", unit="epoch")
    for epoch in epoch_bar:
        epoch_loss = 0.0
        n_batches = 0
        batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{NUM_EPOCHS}",
                         leave=False, unit="batch")
        for batch in batch_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss, _, _ = model(batch, training=True)
            loss.backward()
            optimizer.step()
            scheduler.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss
            n_batches += 1
            batch_bar.set_postfix(batch_loss=f"{batch_loss:.4f}")

            log_row(log_path, timestamp=datetime.datetime.now().isoformat(),
                    phase="train", day=day, epoch=epoch + 1,
                    batch=n_batches, loss=f"{batch_loss:.6f}")

        avg_epoch_loss = epoch_loss / max(n_batches, 1)
        epoch_bar.set_postfix(epoch_loss=f"{avg_epoch_loss:.4f}")
        print(f"    Epoch {epoch+1}/{NUM_EPOCHS}  loss={avg_epoch_loss:.4f}")


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
def evaluate(model, eval_loader, log_path=None, day=None):
    model.eval()
    accum = defaultdict(lambda: [0.0, 0])
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="  Evaluating", leave=False, unit="batch"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, logits, mlm_mask = model(batch, training=False)

            batch_loss = loss.item()
            total_loss += batch_loss
            n_batches += 1

            metrics = compute_ranking_metrics(
                logits, batch["labels"], mlm_mask
            )

            batch_metrics = {k: s / max(c, 1) for k, (s, c) in metrics.items()}

            if log_path is not None:
                log_row(log_path,
                        timestamp=datetime.datetime.now().isoformat(),
                        phase="eval", day=day, batch=n_batches,
                        loss=f"{batch_loss:.6f}",
                        **{k: f"{v:.6f}" for k, v in batch_metrics.items()})

            for k, (s, c) in metrics.items():
                accum[k][0] += s
                accum[k][1] += c

    avg = {k: v[0] / max(v[1], 1) for k, v in accum.items()}
    avg["loss"] = total_loss / max(n_batches, 1)
    return avg


def wipe_memory():
    gc.collect()

## Sliding-Window Training

In [ ]:
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
log_path = init_log(f"training_log_cpu_{time_stamp}.csv")
print(f"Logging to: {log_path}")

for time_index in range(START_WINDOW, FINAL_WINDOW):
    train_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{time_index:04d}/train.parquet")
    )
    eval_day = time_index + 1
    eval_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{eval_day:04d}/valid.parquet")
    )
    if not train_paths:
        print(f"Day {time_index}: no train data, skipping.")
        continue

    print("=" * 50)
    print(f"Day {time_index}: training  (eval on day {eval_day})")
    print("=" * 50)

    train_ds = SessionDataset(train_paths)
    train_loader = DataLoader(
        train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = LambdaLR(
        optimizer, lr_lambda=lambda step: max(0.0, 1 - step / max(total_steps, 1))
    )

    train_one_day(model, train_loader, optimizer, scheduler,
                  log_path, day=time_index)

    if eval_paths:
        eval_ds = SessionDataset(eval_paths)
        eval_loader = DataLoader(
            eval_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
        )
        eval_metrics = evaluate(model, eval_loader,
                                log_path=log_path, day=eval_day)
        print(f"  Eval results (day {eval_day}):")
        for k in sorted(eval_metrics):
            print(f"    {k} = {eval_metrics[k]:.6f}")
    else:
        print(f"  No eval data for day {eval_day}.")

    wipe_memory()

print("\nTraining complete.")

## Final Evaluation on Test Splits

In [ ]:
results = []

for day in range(START_WINDOW, FINAL_WINDOW + 1):
    test_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{day:04d}/test.parquet")
    )
    if not test_paths:
        continue
    test_ds = SessionDataset(test_paths)
    test_loader = DataLoader(
        test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    )
    metrics = evaluate(model, test_loader, log_path=log_path, day=day)
    metrics["time_index"] = day
    results.append(metrics)
    print(f"Day {day}:", {k: f"{v:.4f}" for k, v in metrics.items() if k != "time_index"})

results_df = pd.DataFrame(results).set_index("time_index")
results_df.loc["overall"] = results_df.mean()

print("\n" + "=" * 60)
print("Results Summary:")
print(results_df.to_string())

results_df.to_csv(f"evaluation_results_cpu_{time_stamp}.csv")
print(f"\nSaved to evaluation_results_cpu_{time_stamp}.csv")
print(f"Full training log: {log_path}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

df_plot = results_df.drop("overall", errors="ignore").copy()
df_plot.index = pd.to_numeric(df_plot.index)

metric_configs = {
    "ndcg_at_20":   {"label": "NDCG @ 20",   "ls": "-",  "marker": "o"},
    "ndcg_at_40":   {"label": "NDCG @ 40",   "ls": "--", "marker": "s"},
    "recall_at_20": {"label": "Recall @ 20", "ls": "-.", "marker": "^"},
    "recall_at_40": {"label": "Recall @ 40", "ls": ":",  "marker": "v"},
}

plt.figure(figsize=(10, 6))
for col, cfg in metric_configs.items():
    if col in df_plot.columns:
        plt.plot(df_plot.index, df_plot[col],
                 label=cfg["label"], linestyle=cfg["ls"],
                 marker=cfg["marker"], markersize=5)

plt.legend(title="Evaluation Metrics", bbox_to_anchor=(1.05, 1),
           loc="upper left", borderaxespad=0.0)
plt.xlabel("Day")
plt.ylabel("Metric")
plt.title("Evaluation Metrics on Test Data Over Time (CPU)")
plt.tight_layout()
plt.savefig(f"evaluation_metrics_over_time_cpu_{time_stamp}.png", bbox_inches="tight")
plt.show()

## Save Model

In [ ]:
model_path = os.path.join(os.getcwd(), "saved_model")
os.makedirs(model_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_path, "model.pt"))
print(f"Model saved to {model_path}/model.pt")

## Production-Style Inference from test.pkl

In [ ]:
import pickle

# ---------------------------------------------------------------------------
# 1. Load test.pkl
# ---------------------------------------------------------------------------
TEST_PKL_PATH = os.path.join(os.getcwd(), "test.pkl")
with open(TEST_PKL_PATH, "rb") as f:
    test_df = pickle.load(f)
print(f"Loaded {len(test_df)} sessions from {TEST_PKL_PATH}")

# ---------------------------------------------------------------------------
# 2. Truncate: remove last event, store its pid as ground truth
# ---------------------------------------------------------------------------
ground_truths = []
truncated_rows = []

for idx, row in test_df.iterrows():
    # Extract raw sequences
    cat_seqs = {}
    for feat_name in CATEGORICAL_FEATURES:
        seq = list(row[feat_name])
        cat_seqs[feat_name] = seq
    cont_seqs = {}
    for feat_name in CONTINUOUS_FEATURES:
        cont_seqs[feat_name] = list(row[feat_name])

    orig_len = len(cat_seqs[ITEM_ID_FEATURE])
    if orig_len < 2:
        # Need at least 2 events to truncate one and keep a history
        continue

    # Ground truth = last pid before truncation
    ground_truths.append(cat_seqs[ITEM_ID_FEATURE][-1])

    # Actually truncate — remove the last element from every sequence
    trunc = {}
    for feat_name in CATEGORICAL_FEATURES:
        trunc[feat_name] = cat_seqs[feat_name][:-1]
    for feat_name in CONTINUOUS_FEATURES:
        trunc[feat_name] = cont_seqs[feat_name][:-1]
    trunc["sess_seq_len"] = orig_len - 1

    # Keep any extra columns from the original dataframe
    for col in test_df.columns:
        if col not in trunc:
            trunc[col] = row[col]

    truncated_rows.append(trunc)

trunc_df = pd.DataFrame(truncated_rows).reset_index(drop=True)
ground_truths = np.array(ground_truths, dtype=np.int64)

print(f"Sessions after truncation: {len(trunc_df)}")
print(f"Skipped (length < 2): {len(test_df) - len(trunc_df)}")

# ---------------------------------------------------------------------------
# 3. Build tensors for inference (same logic as SessionDataset.__getitem__)
# ---------------------------------------------------------------------------
def df_to_batch(df, max_seq_len=MAX_SEQ_LEN):
    """Convert a DataFrame of truncated sessions into a batched tensor dict."""
    all_cat = {f: [] for f in CATEGORICAL_FEATURES}
    all_cont = []
    all_mask = []

    for _, row in df.iterrows():
        seq_len = min(int(row["sess_seq_len"]), max_seq_len)

        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[:max_seq_len]
            padded = np.zeros(max_seq_len, dtype=np.int64)
            padded[:len(arr)] = arr
            all_cat[feat_name].append(padded)

        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[:max_seq_len]
            padded = np.zeros(max_seq_len, dtype=np.float32)
            padded[:len(arr)] = arr
            cont_arrays.append(padded)
        all_cont.append(np.stack(cont_arrays, axis=-1))

        mask = np.zeros(max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        all_mask.append(mask)

    batch = {}
    for feat_name in CATEGORICAL_FEATURES:
        batch[feat_name] = torch.tensor(np.stack(all_cat[feat_name]), dtype=torch.long)
    batch["continuous"] = torch.tensor(np.stack(all_cont), dtype=torch.float32)
    batch["attention_mask"] = torch.tensor(np.stack(all_mask), dtype=torch.float32)
    batch["labels"] = batch[ITEM_ID_FEATURE].clone()
    return batch

# ---------------------------------------------------------------------------
# 4. Run inference in mini-batches
# ---------------------------------------------------------------------------
model.eval()
all_predicted_pids = []

for start in tqdm(range(0, len(trunc_df), EVAL_BATCH_SIZE), desc="Inference"):
    end = min(start + EVAL_BATCH_SIZE, len(trunc_df))
    chunk_df = trunc_df.iloc[start:end]
    batch = df_to_batch(chunk_df)
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        _, logits, mlm_mask = model(batch, training=False)

    # mlm_mask marks the last valid position per session — extract that logit
    for i in range(logits.size(0)):
        pos = mlm_mask[i].nonzero(as_tuple=False)
        if len(pos) > 0:
            pred_logit = logits[i, pos[-1].item()]
        else:
            # fallback: use last valid position from attention mask
            last = int(batch["attention_mask"][i].sum().item()) - 1
            pred_logit = logits[i, max(last, 0)]
        predicted_pid = pred_logit.argmax().item()
        all_predicted_pids.append(predicted_pid)

all_predicted_pids = np.array(all_predicted_pids, dtype=np.int64)

# ---------------------------------------------------------------------------
# 5. Build final analysis DataFrame
# ---------------------------------------------------------------------------
# Flatten sequence columns into the output for ad-hoc analysis
analysis_rows = []
for i, (_, row) in enumerate(trunc_df.iterrows()):
    r = {}
    # Categorical sequences (truncated)
    for feat_name in CATEGORICAL_FEATURES:
        r[feat_name] = row[feat_name]
    # Continuous sequences (truncated)
    for feat_name in CONTINUOUS_FEATURES:
        r[feat_name] = row[feat_name]
    # Sequence length after truncation
    r["sess_seq_len"] = row["sess_seq_len"]
    # Any extra columns from the original data
    for col in trunc_df.columns:
        if col not in r:
            r[col] = row[col]
    # Ground truth and prediction
    r["ground_truth_pid"] = ground_truths[i]
    r["predicted_pid"] = all_predicted_pids[i]
    r["correct"] = ground_truths[i] == all_predicted_pids[i]
    analysis_rows.append(r)

inference_df = pd.DataFrame(analysis_rows)

# Quick accuracy summary
n_correct = inference_df["correct"].sum()
accuracy = n_correct / len(inference_df)
print(f"\nExact-match accuracy: {n_correct}/{len(inference_df)} ({accuracy:.4%})")
print(f"\nInference DataFrame shape: {inference_df.shape}")
print(inference_df[["sess_seq_len", "ground_truth_pid", "predicted_pid", "correct"]].head(10))

# Save for ad-hoc analysis
inference_df.to_pickle("inference_results.pkl")
print(f"\nSaved inference results to inference_results.pkl")